In [2]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data" / "raw"

print("Project root:", ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

print("\nAvailable partitions:")
for p in DATA_DIR.iterdir():
    print(p.name)

Project root: C:\Users\S SHREYAS KUMAR\Downloads\ML_HACKATHON\PathSnap
Dataset directory: C:\Users\S SHREYAS KUMAR\Downloads\ML_HACKATHON\PathSnap\data\raw
Dataset exists: True

Available partitions:
student_test
train


In [3]:

from pathlib import Path
import pandas as pd

FILE_TYPES = ["track", "nodes", "arcs", "route"]
PARTITIONS = ["train", "student_test"]

records = []

for partition in PARTITIONS:

    partition_dir = DATA_DIR / partition

    if not partition_dir.exists():
        print(f"Missing partition: {partition}")
        continue

    trips = sorted(
        p for p in partition_dir.iterdir()
        if p.is_dir()
    )

    print(f"\n{partition}: {len(trips)} trips")

    for trip in trips:

        for ext in FILE_TYPES:

            file_path = trip / f"{trip.name}.{ext}"

            exists = file_path.exists()
            row_count = 0
            column_count = None
            file_size = 0

            if exists:

                file_size = file_path.stat().st_size

                with open(file_path, "r", encoding="utf-8") as f:

                    for line in f:
                        if line.strip():
                            row_count += 1

                            if column_count is None:
                                column_count = len(line.split())

            records.append({
                "partition": partition,
                "trip_id": trip.name,
                "file_type": ext,
                "exists": exists,
                "rows": row_count,
                "columns": column_count,
                "size_bytes": file_size
            })

inventory = pd.DataFrame(records)

OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inventory.to_csv(
    OUTPUT_DIR / "dataset_inventory.csv",
    index=False
)

print("\n========== INVENTORY SUMMARY ==========")

summary = inventory.groupby(
    ["partition", "file_type"]
).agg(
    file_count=("exists", "sum"),
    total_rows=("rows", "sum"),
    total_size_bytes=("size_bytes", "sum")
)

display(summary)

print("\nMissing files:")
display(inventory[inventory["exists"] == False])

print("\nInventory saved successfully!")



train: 80 trips

student_test: 10 trips

========== INVENTORY SUMMARY ==========


file_count  total_rows  total_size_bytes
partition    file_type                                          
student_test arcs               10      185041           1902157
             nodes              10       90689           1822179
             route              10        3477             18344
             track              10       19160            625911
train        arcs               80     3876904          44267751
             nodes              80     1933485          38431647
             route              80       31336            165702
             track              80      208417           6872352


Missing files:


,partition,trip_id,file_type,exists,rows,columns,size_bytes



Inventory saved successfully!
